# Final Clustering Robustness

Recomputes the headline reduced-form estimates with the baseline leader-spell clustering and with country-level clustering. The output feeds the clustering robustness table in the final report.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.gid_utils import valid_gid_mask

DATA = ROOT / "data"
PLAD_PATH = DATA / "political leaders" / "PLAD_April_2024.dta"
DMSP_NTL = DATA / "nightlights_adm2_panel.parquet"
HARMONIZED_NTL = DATA / "nightlights_adm2_green_favoritism_panel.parquet"
NO2_PANEL = DATA / "no2_adm2_acag_panel.parquet"
OUT = DATA / "clustering_robustness_results.csv"

In [ ]:
def load_plad():
    plad = pd.read_stata(PLAD_PATH)
    plad = plad[plad["foreign_leader"] == "0"].copy()
    birth_gid = "gid_2" if "gid_2" in plad.columns else "gid_1"
    plad = plad.loc[valid_gid_mask(plad[birth_gid])].copy()
    plad["startyear"] = plad["startyear"].astype(int)
    plad["endyear"] = plad["endyear"].astype(int)
    plad = plad[plad["archigos_id"].str.strip() != "."].copy()
    plad = plad.sort_values(["gid_0", "startyear"]).reset_index(drop=True)

    fixed = plad.copy().reset_index(drop=True)
    for _, group in fixed.groupby("gid_0"):
        idxs = group.index.tolist()
        for i in range(len(idxs) - 1):
            curr, nxt = idxs[i], idxs[i + 1]
            if fixed.loc[curr, "endyear"] >= fixed.loc[nxt, "startyear"]:
                fixed.loc[curr, "endyear"] = fixed.loc[nxt, "startyear"] - 1
    return fixed, birth_gid


PLAD, BIRTH_GID = load_plad()


def build_panel(ntl_path, start, end, include_no2=False):
    base = pd.read_parquet(ntl_path)
    base = base[(base["year"] >= start) & (base["year"] <= end)].copy()
    base = base.loc[valid_gid_mask(base["GID_2"])].copy()
    base = base.dropna(subset=["ntl_mean"]).copy()

    rows = []
    for idx, row in PLAD.iterrows():
        spell_start = max(int(row["startyear"]), start)
        spell_end = min(int(row["endyear"]), end)
        if spell_start > spell_end:
            continue
        for year in range(spell_start, spell_end + 1):
            rows.append({"GID_2": row[BIRTH_GID], "GID_0": row["gid_0"], "year": year, "spell_id": idx})

    leader_years = pd.DataFrame(rows)
    leader_years = leader_years.loc[valid_gid_mask(leader_years["GID_2"])].copy()
    if BIRTH_GID == "gid_1":
        adm1_to_adm2 = base[["GID_2", "GID_1"]].drop_duplicates()
        leader_years = (
            leader_years.rename(columns={"GID_2": "GID_1"})
            .merge(adm1_to_adm2, on="GID_1", how="inner")
            [["GID_2", "GID_0", "year", "spell_id"]]
        )

    leader_years = leader_years.drop_duplicates(subset=["GID_2", "year"])
    leader_years["birth_region_leader"] = 1
    spell_map = leader_years[["GID_0", "year", "spell_id"]].drop_duplicates(subset=["GID_0", "year"])

    panel = base.merge(leader_years[["GID_2", "year", "birth_region_leader"]], on=["GID_2", "year"], how="left")
    panel["birth_region_leader"] = panel["birth_region_leader"].fillna(0).astype(int)
    panel = panel.merge(spell_map, on=["GID_0", "year"], how="left")
    panel["spell_id"] = panel["spell_id"].fillna(panel["GID_0"] + "_" + panel["year"].astype(str) + "_noleader").astype(str)

    if include_no2:
        no2 = pd.read_parquet(NO2_PANEL)
        no2 = no2[(no2["year"] >= start) & (no2["year"] <= end)].copy()
        no2 = no2.loc[valid_gid_mask(no2["GID_2"])].copy()
        no2 = no2.dropna(subset=["no2_mean"]).copy()
        panel = panel.merge(no2[["GID_2", "year", "no2_mean"]], on=["GID_2", "year"], how="inner")
        panel["ln_no2"] = np.log(panel["no2_mean"] + 0.01)
        panel["no2_intensity"] = panel["ln_no2"] - np.log(panel["ntl_mean"] + 0.01)

    panel["ln_ntl"] = np.log(panel["ntl_mean"] + 0.01)
    panel["country_year"] = panel["GID_0"] + "_" + panel["year"].astype(str)
    return panel


def estimate(panel, outcome, treatment="birth_region_leader"):
    d = panel.dropna(subset=[outcome, treatment, "country_year", "spell_id", "GID_0"]).copy()
    idx = d.set_index(["GID_2", "year"])
    model = PanelOLS.from_formula(
        f"{outcome} ~ {treatment} + EntityEffects",
        data=idx,
        other_effects=idx["country_year"],
        drop_absorbed=True,
    )
    leader = model.fit(cov_type="clustered", clusters=idx["spell_id"])
    country = model.fit(cov_type="clustered", clusters=idx["GID_0"])
    return {
        "coef": float(leader.params[treatment]),
        "leader_spell_se": float(leader.std_errors[treatment]),
        "leader_spell_p": float(leader.pvalues[treatment]),
        "country_se": float(country.std_errors[treatment]),
        "country_p": float(country.pvalues[treatment]),
        "nobs": int(leader.nobs),
        "leader_spell_clusters": int(d["spell_id"].nunique()),
        "country_clusters": int(d["GID_0"].nunique()),
        "treated": int(d[treatment].sum()),
    }

In [ ]:
specs = [
    ("Nightlights, DMSP 1992--2013", DMSP_NTL, 1992, 2013, "ln_ntl", False),
    ("Nightlights, DMSP 2005--2013", DMSP_NTL, 2005, 2013, "ln_ntl", False),
    ("Nightlights, harmonized 2005--2019", HARMONIZED_NTL, 2005, 2019, "ln_ntl", False),
    ("NO2, 2005--2019", HARMONIZED_NTL, 2005, 2019, "ln_no2", True),
    ("NO2 intensity, 2005--2019", HARMONIZED_NTL, 2005, 2019, "no2_intensity", True),
]

rows = []
for label, path, start, end, outcome, include_no2 in specs:
    panel = build_panel(path, start, end, include_no2=include_no2)
    row = {"spec": label, "outcome": outcome, "start": start, "end": end}
    row.update(estimate(panel, outcome))
    rows.append(row)

out = pd.DataFrame(rows)
out.to_csv(OUT, index=False)
out